<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [7]</a>'.</span>

In [1]:
# Parameters
SEED = 42
DOWNSAMPLE_SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-05
EPOCHS = 3
OUTPUT_DIR = "models/beto"


# 02 — BETO fine-tuning (Spanish BERT, dccuchile/bert-base-spanish-wwm-cased)
**Autores:** Giuliano Crenna, Bruno Emmanuel Pace (UGR)  
**Etapa:** 4 — Modelado  
**Hardware:** GPU local RTX 3060 Ti 8GB VRAM  
**Entorno:** system Python (CUDA), kernel `python3 (system-CUDA)`  

BETO es la versión de BERT pre-entrenada en español por [dccuchile](https://huggingface.co/dccuchile/bert-base-spanish-wwm-cased). 110M parámetros, Whole-Word-Masking, cased.

**Setup:** train balanceado por downsampling (424.436 filas), 3 épocas, lr=2e-5, batch=16, fp16 (activado automáticamente con CUDA), EarlyStopping(patience=2). Hiperparámetros leídos de `configs/models.yaml`.

**Comparación esperada vs baseline:**
- baseline `logreg_full_balanced` test: F1-macro=0.655, AUC depressive=0.755
- BETO target: F1-macro ≥0.72, AUC depressive ≥0.82 (según literatura sobre datasets similares en español)

In [2]:
# Parámetros (papermill overrides con -p SEED ... -p EPOCHS ... -p BATCH_SIZE ...)
SEED = 42
DOWNSAMPLE_SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 3
OUTPUT_DIR = "models/beto"

In [3]:
import os, sys, time
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

assert torch.cuda.is_available(), (
    "CUDA no disponible. Este notebook requiere GPU. "
    f"torch={torch.__version__}, cuda={torch.cuda.is_available()}"
)
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"torch: {torch.__version__} | cuda: {torch.__version__.split('+')[-1]}")

from src.utils.seeds import set_seed
from src.models.train_baseline import load_split_data
from src.models.train_beto import run_training
from src.models.predict import predict_texts
from src.evaluation.metrics import compute
from sklearn.metrics import confusion_matrix, roc_auc_score

set_seed(SEED)
OUTPUT_DIR = REPO_ROOT / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = REPO_ROOT / "reports" / "figures"
TAB_DIR = REPO_ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROC = REPO_ROOT / "data" / "processed" / "splits"
print(f"Repo: {REPO_ROOT}")
print(f"Output: {OUTPUT_DIR}")

GPU: NVIDIA GeForce RTX 3060 Ti | VRAM: 8.6 GB
torch: 2.14.0+cu126 | cuda: cu126


Repo: D:\Github\Deteccion-Temprana-Depresion
Output: D:\Github\Deteccion-Temprana-Depresion\models\beto


## 1. Carga de splits user-level (Coello-Guilarte)

In [4]:
assert (DATA_PROC / "train.parquet").exists()
df_train, df_val, df_test = load_split_data(data_dir=DATA_PROC, max_samples=None, seed=SEED)

print(f"Train: {len(df_train):,} | {df_train['label'].value_counts().to_dict()}")
print(f"Val:   {len(df_val):,} | {df_val['label'].value_counts().to_dict()}")
print(f"Test:  {len(df_test):,} | {df_test['label'].value_counts().to_dict()}")
assert len(df_train) == 700717 and len(df_val) == 95630 and len(df_test) == 250847
print("OK — conteos match con split_manifest.json")

2026-09-29 17:46:46 | INFO    | train_baseline | Cargando splits desde D:\Github\Deteccion-Temprana-Depresion\data\processed\splits...


Train: 700,717 | {0: 488499, 2: 212218}
Val:   95,630 | {0: 77425, 2: 18205}
Test:  250,847 | {0: 180347, 2: 70500}
OK — conteos match con split_manifest.json


## 2. Downsampling del train (50/50)

BETO se entrena sobre train balanceado para evitar weighted loss / sampler. Val y test se mantienen intactos (distribución real).

In [5]:
df_pos = df_train[df_train["label"] == 2]
df_neg = df_train[df_train["label"] == 0].sample(n=len(df_pos), random_state=DOWNSAMPLE_SEED)
df_down = (
    pd.concat([df_pos, df_neg])
    .sample(frac=1.0, random_state=DOWNSAMPLE_SEED)
    .reset_index(drop=True)
)
print(f"Downsampled train: {len(df_down):,} | {df_down['label'].value_counts().to_dict()}")
assert len(df_down) == 424436
assert df_down["label"].value_counts().to_dict() == {0: 212218, 2: 212218}

Downsampled train: 424,436 | {2: 212218, 0: 212218}


## 3. Fine-tuning de BETO

Estimación de tiempo en RTX 3060 Ti 8GB (batch=16, max_length=128, fp16):
- Train steps/época: 424.436 / 16 ≈ 26.528
- Tiempo/época: ~15 min
- Eval val cada época: 95.630 / 32 ≈ 2.988 steps, ~2 min
- **Total estimado: 3 epochs × ~17 min ≈ 50 min + checkpoints**

In [6]:
cfg_path = REPO_ROOT / "configs" / "models.yaml"
import yaml
cfg = yaml.safe_load(open(cfg_path, encoding="utf-8")) if cfg_path.exists() else {}
beto_cfg = cfg.get("beto", {})
MODEL_NAME = beto_cfg.get("model_name", "dccuchile/bert-base-spanish-wwm-cased")
print(f"Modelo: {MODEL_NAME}")
print(f"Hiperparámetros: lr={LEARNING_RATE}, batch={BATCH_SIZE}, epochs={EPOCHS}, max_length={MAX_LENGTH}")

t0 = time.time()
trainer, test_metrics = run_training(
    df_train=df_down,
    df_val=df_val,
    df_test=df_test,
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    output_dir=str(OUTPUT_DIR),
    seed=SEED,
)
train_min = (time.time() - t0) / 60
print(f"\nEntrenamiento completo en {train_min:.1f} min")
print(f"Test metrics (Trainer.evaluate): {test_metrics}")

# Persistir log history para curvas
import json
log_history = trainer.state.log_history
with open(OUTPUT_DIR / "trainer_log_history.json", "w", encoding="utf-8") as f:
    json.dump(log_history, f, indent=2, default=str)
print(f"Log history guardado en {OUTPUT_DIR / 'trainer_log_history.json'}")

2026-09-29 17:46:49 | INFO    | train_beto | Dispositivo detectado: CUDA


2026-09-29 17:46:49 | INFO    | train_beto | Etiquetas detectadas: {0: 0, 2: 1} (num_labels=2)


2026-09-29 17:46:49 | INFO    | train_beto | Cargando tokenizer para 'dccuchile/bert-base-spanish-wwm-cased'...


Modelo: dccuchile/bert-base-spanish-wwm-cased
Hiperparámetros: lr=2e-05, batch=16, epochs=3, max_length=128


2026-09-29 17:46:53 | INFO    | train_beto | Cargando modelo 'dccuchile/bert-base-spanish-wwm-cased'...


...

{'train_runtime': 8049.811, 'train_samples_per_second': 158.179, 'train_steps_per_second': 9.886, 'train_loss': 0.3634243835921646, 'epoch': 3.0}


  0%|          | 0/7839 [00:00<?, ?it/s]

2026-09-29 20:06:59 | INFO    | train_beto | Métricas en TEST: {'eval_loss': 0.5767449736595154, 'eval_accuracy': 0.7551934047447248, 'eval_f1_macro': 0.7168936140229227, 'eval_f1_weighted': 0.7624922994410467, 'eval_kappa': 0.43698622408684284, 'eval_runtime': 350.0351, 'eval_samples_per_second': 716.634, 'eval_steps_per_second': 22.395, 'epoch': 3.0}


2026-09-29 20:07:03 | INFO    | train_beto | Modelo final y artefactos guardados en D:\Github\Deteccion-Temprana-Depresion\models\beto/best_model



Entrenamiento completo en 140.2 min
Test metrics (Trainer.evaluate): {'eval_loss': 0.5767449736595154, 'eval_accuracy': 0.7551934047447248, 'eval_f1_macro': 0.7168936140229227, 'eval_f1_weighted': 0.7624922994410467, 'eval_kappa': 0.43698622408684284, 'eval_runtime': 350.0351, 'eval_samples_per_second': 716.634, 'eval_steps_per_second': 22.395, 'epoch': 3.0}
Log history guardado en D:\Github\Deteccion-Temprana-Depresion\models\beto\trainer_log_history.json


## 4. Evaluación post-hoc con el schema unificado

El `trainer.evaluate()` solo emite 4 métricas (acc, f1_macro, f1_weighted, kappa). Para el schema unificado de `all_models_metrics.csv` necesitamos también **AUC binario depressive** y **F1 por clase**, así que re-evaluamos con `predict.py` + `compute()`.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [7]:
best_model_dir = OUTPUT_DIR / "best_model"
rows = []
for split_name, df_split in [("val", df_val), ("test", df_test)]:
    t0 = time.time()
    preds = predict_texts(str(best_model_dir), df_split["text_clean"].astype(str).tolist())
    print(f"  {split_name}: predict sobre {len(preds):,} ejemplos en {time.time()-t0:.1f}s")
    y_true = df_split["label"].astype(int).tolist()
    y_pred = [p["label_pred"] for p in preds]
    y_proba = [p["probabilities"] for p in preds]
    m = compute(y_true, y_pred, y_proba=y_proba)

    # AUC binario: la columna con label 2 (depressive) en y_proba
    # predict.py devuelve probabilities según id2label; con label2id={0:0, 2:1} → columna 1 = depressive
    proba_arr = np.array(y_proba)
    if proba_arr.shape[1] == 2:
        auc_bin = float(roc_auc_score(y_true, proba_arr[:, 1]))
    else:
        # fallback: buscar la columna con mayor label original
        auc_bin = float("nan")

    rows.append({
        "model": "beto",
        "split": split_name,
        "n_train_rows": len(df_down),
        "class_balance_strategy": "downsample_majority",
        "feature_set": "beto_transformer",
        "accuracy": m["accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],
        "kappa": m["kappa"],
        "auc_binary_depressive": auc_bin,
        "f1_class_0_control": m["report"].get("0", {}).get("f1-score", np.nan),
        "f1_class_2_depressive": m["report"].get("2", {}).get("f1-score", np.nan),
        "created_at": pd.Timestamp.utcnow().isoformat(timespec="seconds"),
    })

df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))

OutOfMemoryError: CUDA out of memory. Tried to allocate 26.54 GiB. GPU 0 has a total capacity of 8.00 GiB of which 4.80 GiB is free. Process 10484 has 17179869184.00 GiB memory in use. Process 10072 has 17179869184.00 GiB memory in use. Process 13768 has 17179869184.00 GiB memory in use. Process 1644 has 17179869184.00 GiB memory in use. Process 5008 has 17179869184.00 GiB memory in use. Process 29680 has 17179869184.00 GiB memory in use. Process 9440 has 17179869184.00 GiB memory in use. Process 5508 has 17179869184.00 GiB memory in use. Process 5604 has 17179869184.00 GiB memory in use. Process 23984 has 17179869184.00 GiB memory in use. Process 19072 has 17179869184.00 GiB memory in use. Process 20288 has 17179869184.00 GiB memory in use. Process 38436 has 17179869184.00 GiB memory in use. Process 42052 has 17179869184.00 GiB memory in use. Process 44328 has 17179869184.00 GiB memory in use. Process 41892 has 17179869184.00 GiB memory in use. Process 30528 has 17179869184.00 GiB mem

## 5. Persistencia de artefactos

In [ ]:
unified_csv = TAB_DIR / "all_models_metrics.csv"
df_existing = pd.read_csv(unified_csv)
df_combined = pd.concat([df_existing, df_metrics], ignore_index=True)
df_combined.to_csv(unified_csv, index=False)
print(f"Métricas unificadas actualizadas: {unified_csv} ({len(df_combined)} filas)")

assert df_metrics["auc_binary_depressive"].notna().all(), "AUC binario no debe ser NaN"
assert (df_metrics["split"].isin(["val", "test"])).all(), "split debe ser val o test"
print("OK — artefactos consistentes")

## 6. Visualizaciones: confusion matrix + training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for idx, (split_name, df_split) in enumerate([("val", df_val), ("test", df_test)]):
    preds = predict_texts(str(best_model_dir), df_split["text_clean"].astype(str).tolist())
    y_true = df_split["label"].astype(int).tolist()
    y_pred = [p["label_pred"] for p in preds]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 2])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=[0, 2], yticklabels=[0, 2])
    axes[idx].set_title(f"BETO — {split_name}")
    axes[idx].set_xlabel("Predicho")
    axes[idx].set_ylabel("Real")
plt.tight_layout()
fig_path = FIG_DIR / "beto_confusion_matrix.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Figura guardada en {fig_path}")

In [ ]:
# Training curves desde log_history
train_steps, train_loss = [], []
eval_epochs, eval_loss, eval_f1 = [], [], []
for entry in log_history:
    if "loss" in entry and "epoch" in entry and "eval_loss" not in entry:
        train_steps.append(entry.get("step", 0))
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        eval_epochs.append(entry["epoch"])
        eval_loss.append(entry["eval_loss"])
        eval_f1.append(entry.get("eval_f1_macro", np.nan))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(train_steps, train_loss, color="#5bc0de", label="train loss")
axes[0].plot(eval_epochs, eval_loss, color="#d9534f", marker="o", label="val loss")
axes[0].set_xlabel("Step / Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("BETO — training & validation loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(eval_epochs, eval_f1, color="#5cb85c", marker="o", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1-macro (val)")
axes[1].set_title("BETO — validation F1-macro per epoch")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig_path = FIG_DIR / "beto_training_curves.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Figura guardada en {fig_path}")

In [ ]:
print("\n=== Leaderboard (test) ===")
df_all = pd.read_csv(TAB_DIR / "all_models_metrics.csv")
df_test_lb = df_all[df_all["split"] == "test"].sort_values("f1_macro", ascending=False)
print(df_test_lb[["model","feature_set","f1_macro","auc_binary_depressive","f1_class_2_depressive"]].to_string(index=False))

## 7. Conclusiones

- **BETO fine-tuned** sobre train downsampled 424k (50/50) durante 3 épocas, lr=2e-5, batch=16, fp16 en RTX 3060 Ti.
- **Tiempo total**: ~`{train_min:.1f}` min (registrado en el log).
- **Artefactos**: `models/beto/best_model/{config.json, pytorch_model.bin, tokenizer.*}` + `model_meta.json` + `trainer_log_history.json`.
- **Métricas unificadas**: append 2 filas a `reports/tables/all_models_metrics.csv` (val + test).
- **Figuras**: `reports/figures/beto_confusion_matrix.png` + `beto_training_curves.png`.
- **Comparación**: BETO vs baseline TF-IDF+LogReg.